# Data Preparation

Merges GSWA HyLogger data with WAMEX assays into one dataset, cleans it and builds aligned SWIR, TIR and combined DataFrames for comparison. No train/test split is done here.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "ml_pipeline" else Path.cwd()

DATA_DIR = ROOT / "data"
CSV_DIR = DATA_DIR / "INTERIM_HyLogger_Dataset_Package_GDA2020_DB" / "DATABASES" / "CSV"
MINERAL_LOGS_DIR = DATA_DIR / "extracted_data" / "mineral_logs"
MANIFEST = DATA_DIR / "extracted_data" / "mineral_logs_manifest.csv"
RANDOM_STATE = 67

### Mineral logs + Au assays

Each mineral-log row is matched to `WAMEX_AssayFlatData` records with the same `CollarId` (resolved via `HyLogger_Dataset_Index`) whose `FromDepth` **and** `ToDepth` both agree with the log row's interval within an absolute tolerance (`DEPTH_ATOL_M`, an assumed recording tolerance). There is no relative tolerance and depths are never rounded, so an assay that merely contains a log row (e.g. a 2 m composite over a 1 m log row) does not match. The realised boundary differences are reported below.

All numeric inputs are converted once at read time. Malformed text and infinities become NaN and are listed in `input_issues`; legitimate `N.A.` values stay NaN.

The dataset represents **measured, nonnegative Au results**. Excluded, and counted per hole below: invalid depth intervals, log rows without a matching assay, missing or nonfinite `Au_PPM`, ambiguous matches, and negative `Au_PPM` (typically below-detection-limit codes, so such samples are not represented).

A log row with several matching assays is kept only if all candidates are identical records (same boundaries, `Au_PPM` and `CompanySampleId`); the lowest `AssayFlatDataId` is kept and the rest counted as duplicates. If candidates differ in any of these fields, including one having missing Au, the row is excluded as `conflicting_assays` rather than resolved arbitrarily.

`CollarId`, `Project`, `AssayFlatDataId` and `CompanySampleId` are metadata, not predictors. `Dataset_ID` is the first column.

In [ ]:
import importlib
import sys

sys.path.insert(0, str(ROOT / "src"))
import gswa_hackathon.build_dataset as build_module

# Pick up edits to src/ without restarting the kernel.
build_module = importlib.reload(build_module)
DEPTH_ATOL_M = build_module.DEPTH_ATOL_M
assert_aligned = build_module.assert_aligned
build_dataset = build_module.build_dataset
common_cohort_mask = build_module.common_cohort_mask

pd.set_option("display.max_columns", None)

df, exclusion_report, input_issues = build_dataset()
print(df.shape)
df.head()

# Malformed or infinite numeric inputs are reported as NaN.
display(
    input_issues
    if not input_issues.empty
    else "no malformed or infinite numeric inputs"
)

# Exclusion counts are reported per hole and project.
display(exclusion_report)
display(exclusion_report.groupby("Project").sum(numeric_only=True))

retained_share = exclusion_report["retained"].sum() / exclusion_report["log_rows"].sum()
print(f"Boundary matching retained {retained_share:.1%} of mineral-log rows")
no_matches = exclusion_report.index[exclusion_report["retained"] == 0].tolist()

if no_matches:
    print("WARNING: no matching samples for", no_matches)

assert (df["Au_PPM"] >= 0).all() and np.isfinite(df["Au_PPM"]).all()

## Realised boundary differences between log and assay intervals (m)
boundary_diff = pd.concat(
    [
        (df["Depth_from_(m)"] - df["Assay_FromDepth"]).abs(),
        (df["Depth_to_(m)"] - df["Assay_ToDepth"]).abs(),
    ],
    axis=1,
).max(axis=1)
display(boundary_diff.describe())

for tolerance in (0.0, 1e-6, 1e-3, 1e-2):
    print(f"pairs within {tolerance:g} m: {(boundary_diff <= tolerance).sum()}")
print(f"DEPTH_ATOL_M = {DEPTH_ATOL_M:g} m")

### Columns and common comparison cohort

The cohort is established once: samples with at least one usable (finite) SWIR descriptor **and** at least one usable TIR descriptor. Partially missing descriptors are kept as NaN (no imputation, scaling or zero-filling). Only samples with no usable SWIR or no usable TIR descriptor are left out of this comparison. `df_swir`, `df_tir` and `df_combined` are all derived from the same rows.

In [ ]:
ID_COLUMNS = [
    "Dataset_ID",
    "Depth_from_(m)",
    "Depth_to_(m)",
    "Assay_FromDepth",
    "Assay_ToDepth",
]
METADATA_COLUMNS = ["CollarId", "Project", "AssayFlatDataId", "CompanySampleId"]
TARGET = "Au_PPM"

SPECTRAL_FEATURES_SWIR = [
    "900D_Fe",
    "900W_Fe",
    "1900D_Water",
    "2200D_AlOH",
    "2200W_AlOH",
    "2250D_FeOH",
    "2250W_FeOH",
]
SPECTRAL_FEATURES_TIR = ["8625D_Quartz", "11300D_Carbonate", "11300W_Carbonate"]

# raises if descriptor is not numeric or infinite(NaN is allowed)
in_cohort = common_cohort_mask(df, SPECTRAL_FEATURES_SWIR, SPECTRAL_FEATURES_TIR)

has_swir = np.isfinite(df[SPECTRAL_FEATURES_SWIR]).any(axis=1)
has_tir = np.isfinite(df[SPECTRAL_FEATURES_TIR]).any(axis=1)
per_hole = exclusion_report.copy()
per_hole["no_usable_swir"] = (
    df[~has_swir].groupby("Dataset_ID").size().reindex(per_hole.index, fill_value=0)
)
per_hole["no_usable_tir"] = (
    df[~has_tir].groupby("Dataset_ID").size().reindex(per_hole.index, fill_value=0)
)
per_hole["in_cohort"] = (
    df[in_cohort].groupby("Dataset_ID").size().reindex(per_hole.index, fill_value=0)
)
display(per_hole)
display(per_hole.groupby("Project").sum(numeric_only=True))

print(f"common cohort: {in_cohort.sum()} of {len(df)} matched samples")

### DF FOR SWIR

In [ ]:
df_swir = df.loc[
    in_cohort, ID_COLUMNS + METADATA_COLUMNS + [TARGET] + SPECTRAL_FEATURES_SWIR
].reset_index(drop=True)

# missing descriptors remain NaN
print("missing per SWIR descriptor:")
print(df_swir[SPECTRAL_FEATURES_SWIR].isna().sum())

# displays general info
df_swir.info(verbose=True)
display(df_swir.head())
display(df_swir.describe())

### DF FOR TIR 

In [ ]:
df_tir = df.loc[
    in_cohort, ID_COLUMNS + METADATA_COLUMNS + [TARGET] + SPECTRAL_FEATURES_TIR
].reset_index(drop=True)

# Missing descriptors remain NaN
print("missing per TIR descriptor:")
print(df_tir[SPECTRAL_FEATURES_TIR].isna().sum())

# displays general info
df_tir.info(verbose=True)
display(df_tir.head())
display(df_tir.describe())

### DF COMBINED

In [ ]:
df_combined = df.loc[
    in_cohort,
    ID_COLUMNS
    + METADATA_COLUMNS
    + [TARGET]
    + SPECTRAL_FEATURES_SWIR
    + SPECTRAL_FEATURES_TIR,
].reset_index(drop=True)

# same sample keys, row order and Au_PPM targets in all three frames.
assert_aligned([df_swir, df_tir, df_combined])
assert not set(METADATA_COLUMNS + ID_COLUMNS + [TARGET]) & set(
    SPECTRAL_FEATURES_SWIR + SPECTRAL_FEATURES_TIR
)

# displays missing descriptors (NaN) per hole inside the cohort.
display(
    df_combined.groupby("Dataset_ID")[
        SPECTRAL_FEATURES_SWIR + SPECTRAL_FEATURES_TIR
    ].agg(lambda s: s.isna().sum())
)

df_combined.info(verbose=True)
display(df_combined.head())

# Summarize Au and descriptor statistics and save them as an image.
summary = df_combined[
    [TARGET] + SPECTRAL_FEATURES_SWIR + SPECTRAL_FEATURES_TIR
].describe()
display(summary)

fig, ax = plt.subplots(figsize=(20, 3.5))
ax.axis("off")
table = ax.table(
    cellText=summary.map("{:.4f}".format).values,
    rowLabels=summary.index,
    colLabels=summary.columns,
    loc="center",
)

table.auto_set_font_size(False)
table.set_fontsize(9)
table.auto_set_column_width(range(len(summary.columns)))
table.scale(1, 1.5)
fig.savefig(
    ROOT / "ml_pipeline" / "df_combined_describe.png", dpi=200, bbox_inches="tight"
)
plt.show()

## ML export

The combined labelled cohort is exported locally for the modelling notebook.
The output is derived from the validated cohort and remains excluded from Git.

In [ ]:
ML_DATASET_PATH = ROOT / "data" / "extracted_data" / "ml_dataset.csv"
ML_DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
df_combined.to_csv(ML_DATASET_PATH, index=False)
print(f"Wrote {len(df_combined):,} rows to {ML_DATASET_PATH}")